# TrustGuard — Deployment Notebook
## Deliverable 3 · Track A · Final Deployment & Technical Report

**Goal:** Prepare and validate the Streamlit deployment for the AI Fraud Detection System.

This notebook:
1. Verifies all artefacts (models, scaler, ChromaDB, outputs)
2. Validates model loading and inference
3. Validates the RAG pipeline end-to-end
4. Generates a sample XAI (SHAP) plot
5. Runs a smoke-test of `app.py`
6. Produces the deployment-ready file list

## 0. Setup — Install / verify deployment packages

In [1]:
# Run this cell first to confirm the environment is ready.
# On Streamlit Cloud this is handled automatically from requirements.txt.
import subprocess, sys

PACKAGES = [
    'streamlit', 'shap', 'chromadb',
    'sentence_transformers', 'rank_bm25', 'openai',
]

missing = []
for pkg in PACKAGES:
    try:
        __import__(pkg.replace('-','_'))
        print(f'  {pkg}')
    except ImportError:
        missing.append(pkg)
        print(f'  ❌ {pkg}  ← MISSING')

if missing:
    print('\nInstalling missing packages…')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + missing)
    print('Done. Restart kernel if prompted.')
else:
    print('\n All deployment packages present.')

  streamlit
  ❌ shap  ← MISSING
  chromadb
  sentence_transformers
  rank_bm25
  openai

Installing missing packages…
Done. Restart kernel if prompted.


---
## 1. Artefact Verification

In [ ]:
import os, json
import pandas as pd

import os

# Resolve project root regardless of where the notebook is launched from
_NB_DIR = os.path.abspath('')           # current working dir when running
_NB_FILE = os.path.abspath('__file__') if '__file__' in dir() else _NB_DIR

# Go up 2 levels: notebooks/final_deliverable_notebooks/ → notebooks/ → project root
ROOT = os.path.dirname(os.path.dirname(os.path.abspath(_NB_DIR)))

# Safety check — confirm we landed at the right place
assert os.path.exists(os.path.join(ROOT, "outputs")), \
    f"ROOT looks wrong: {ROOT} — expected to find 'outputs/' folder here"
assert os.path.exists(os.path.join(ROOT, "chroma_db")), \
    f"ROOT looks wrong: {ROOT} — expected to find 'chroma_db/' folder here"

print(f"✅ Project ROOT resolved to: {ROOT}")

REQUIRED_FILES = {
    'Deployment model'    : 'outputs/deployment/model.pkl',
    'Deployment scaler'   : 'outputs/deployment/scaler.pkl',
    'Model metadata'      : 'outputs/deployment/model_meta.json',
    'Feature names'       : 'outputs/deployment/feature_names.json',
    'XGBoost model'       : 'outputs/models/xgboost.pkl',
    'Random Forest model' : 'outputs/models/random_forest.pkl',
    'Neural Network model': 'outputs/models/neural_network.pkl',
    'Logistic Reg. model' : 'outputs/models/logistic_regression.pkl',
    'Models scaler'       : 'outputs/models/scaler.pkl',
    'Model comparison'    : 'outputs/metrics/model_comparison.json',
    'XGBoost metrics'     : 'outputs/metrics/xgboost_metrics.json',
    'Ablation results'    : 'outputs/ablation/ablation_results.csv',
    'ChromaDB sqlite'     : 'chroma_db/chroma.sqlite3',
    'RAG module'          : 'rag_module.py',
    'Streamlit app'       : 'app.py',
    'Requirements'        : 'requirements.txt',
}

results = []
all_ok  = True
for label, rel_path in REQUIRED_FILES.items():
    full = os.path.join(ROOT, rel_path)
    ok   = os.path.exists(full)
    size = f"{os.path.getsize(full)/1024:.1f} KB" if ok else '—'
    results.append({'File': label, 'Path': rel_path, 'Status': '✅' if ok else '❌', 'Size': size})
    if not ok:
        all_ok = False

df_artefacts = pd.DataFrame(results)
print(df_artefacts.to_string(index=False))
print()
print('🎉 All artefacts present!' if all_ok else '⚠️  Some files are missing — see ❌ rows above.')

---
## 2. Model Loading & Inference Validation

In [ ]:
import joblib, numpy as np

# Load deployment artefacts
model  = joblib.load(os.path.join(ROOT, 'outputs/deployment/model.pkl'))
scaler = joblib.load(os.path.join(ROOT, 'outputs/deployment/scaler.pkl'))
with open(os.path.join(ROOT, 'outputs/deployment/model_meta.json')) as f:
    meta = json.load(f)

print('Model type :', meta['model_type'])
print('Features   :', meta['n_features'])
print('Test AUC   :', meta['test_auc'])
print('Test Recall:', meta['test_recall'])
print()

In [ ]:
FEATURES = meta['features']

# ── Test Case 1: Known fraud pattern (CASH_OUT draining account) ──────────
fraud_sample = {
    'step': 200, 'amount': 180000.0,
    'oldbalanceOrg': 180000.0, 'newbalanceOrig': 0.0,
    'oldbalanceDest': 0.0, 'newbalanceDest': 0.0,
    'balanceDiff': 180000.0, 'amount_ratio': 1.0,
    'type_CASH_OUT': 1, 'type_DEBIT': 0, 'type_PAYMENT': 0, 'type_TRANSFER': 0,
}

# ── Test Case 2: Normal payment ───────────────────────────────────────────
legit_sample = {
    'step': 50, 'amount': 1500.0,
    'oldbalanceOrg': 50000.0, 'newbalanceOrig': 48500.0,
    'oldbalanceDest': 10000.0, 'newbalanceDest': 11500.0,
    'balanceDiff': 1500.0, 'amount_ratio': 0.03,
    'type_CASH_OUT': 0, 'type_DEBIT': 0, 'type_PAYMENT': 1, 'type_TRANSFER': 0,
}

def predict(sample, label):
    X = np.array([[sample[f] for f in FEATURES]])
    X_s = scaler.transform(X)
    prob = model.predict_proba(X_s)[0, 1]
    verdict = 'FRAUD' if prob >= meta['decision_threshold'] else 'LEGITIMATE'
    print(f'{label}: prob={prob:.4f}  verdict={verdict}')
    return prob, X_s

fraud_prob, fraud_Xs = predict(fraud_sample, '🔴 Fraud sample  ')
legit_prob, legit_Xs = predict(legit_sample, '🟢 Legit sample  ')

assert fraud_prob > 0.5, 'FAIL: fraud sample not flagged!'
assert legit_prob < 0.5, 'FAIL: legit sample flagged as fraud!'
print('\n✅ Inference validation passed.')

---
## 3. XAI — SHAP Explanation (with fallback)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

xai_method = 'unknown'

try:
    try:
        import shap
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "shap"])
        import shap
    explainer   = shap.TreeExplainer(model)
    shap_values = explainer(fraud_Xs)
    sv          = shap_values[0]
    vals        = sv.values
    base        = float(sv.base_values)

    order  = np.argsort(np.abs(vals))[::-1][:12][::-1]
    fnames = [FEATURES[i] for i in order]
    fvals  = vals[order]
    colors = ['#DC2626' if v > 0 else '#16A34A' for v in fvals]

    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.barh(fnames, fvals, color=colors, edgecolor='white')
    ax.axvline(0, color='#263D5B', linewidth=1.2, linestyle='--', alpha=0.5)
    ax.set_xlabel('SHAP Value')
    ax.set_title(f'SHAP Waterfall — Fraud Sample  (base={base:.3f})',
                 color='#263D5B', fontweight='bold')
    for sp in ax.spines.values(): sp.set_visible(False)
    plt.tight_layout()
    plt.savefig('outputs/deployment/xai_shap_sample.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ SHAP explanation generated → outputs/deployment/xai_shap_sample.png')
    xai_method = 'SHAP TreeExplainer'

except Exception as e:
    print(f'SHAP unavailable ({e}). Using feature importance fallback.')
    importances = model.feature_importances_
    order  = np.argsort(importances)
    fig, ax = plt.subplots(figsize=(9, 5))
    palette = plt.cm.Blues(np.linspace(0.3, 0.9, len(importances)))
    ax.barh([FEATURES[i] for i in order], importances[order], color=palette)
    ax.set_title('XGBoost Feature Importance (Gain)', color='#263D5B', fontweight='bold')
    for sp in ax.spines.values(): sp.set_visible(False)
    plt.tight_layout()
    plt.savefig('outputs/deployment/xai_fallback.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Fallback XAI chart generated → outputs/deployment/xai_fallback.png')
    xai_method = 'XGBoost Feature Importance (fallback)'

print(f'XAI method: {xai_method}')

---
## 4. RAG Pipeline Validation

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(ROOT, 'notebooks', 'final_deliverable_notebooks'))

# Patch ChromaDB path BEFORE importing rag_module
import rag_module
rag_module.RAG_CONFIG['CHROMA_DB_PATH'] = os.path.join(os.getcwd(), 'chroma_db')

print('ChromaDB path set to:', rag_module.RAG_CONFIG['CHROMA_DB_PATH'])
print('Exists:', os.path.exists(rag_module.RAG_CONFIG['CHROMA_DB_PATH']))

In [ ]:
# Provide your OpenAI key here for notebook validation.
# On Streamlit Cloud it is read from st.secrets or the sidebar.
import getpass

openai_key = os.environ.get('OPENAI_API_KEY', '')
if not openai_key:
    openai_key = getpass.getpass('Enter OpenAI API key (leave blank to skip RAG test): ')

if not openai_key:
    print('  No key provided — skipping live RAG call.')
    print('   The app will still work; users supply the key via sidebar or Streamlit Secrets.')
else:
    print('Key received. Running RAG pipeline on fraud sample…')
    result = rag_module.rag_pipeline_for_streamlit(
        fraud_probability = fraud_prob,
        features          = fraud_sample,
        transaction_id    = 'TXN-DEPLOY-TEST',
        openai_api_key    = openai_key,
    )
    print()
    print(f'Transaction : {result.transaction_id}')
    print(f'Risk Tier   : {result.risk_tier}')
    print(f'Grounding   : {result.grounding_score:.0%}')
    print(f'Latency     : {result.latency_seconds:.1f}s')
    print(f'No-evidence : {result.no_evidence_flag}')
    print()
    print('── RAG Response (truncated) ──────────────────')
    print(result.response_text[:800], '…')
    print()
    print('── Sources ──────────────────────────────────')
    for s in result.sources:
        print(' •', s)
    print('\n✅ RAG pipeline validated.')

---
## 5. Batch Inference Smoke Test

In [ ]:
import pandas as pd

# Synthetic mini-batch (simulates a user-uploaded CSV)
batch = pd.DataFrame([
    {'step':200,'amount':180000,'oldbalanceOrg':180000,'newbalanceOrig':0,
     'oldbalanceDest':0,'newbalanceDest':0,'type':'CASH_OUT','isFraud':1},
    {'step':50 ,'amount':1500  ,'oldbalanceOrg':50000 ,'newbalanceOrig':48500,
     'oldbalanceDest':10000,'newbalanceDest':11500,'type':'PAYMENT','isFraud':0},
    {'step':312,'amount':75000 ,'oldbalanceOrg':75000 ,'newbalanceOrig':0,
     'oldbalanceDest':0,'newbalanceDest':0,'type':'TRANSFER','isFraud':1},
    {'step':10 ,'amount':500   ,'oldbalanceOrg':10000 ,'newbalanceOrig':9500,
     'oldbalanceDest':500,'newbalanceDest':1000,'type':'PAYMENT','isFraud':0},
])

def prep_batch(df):
    d = df.copy()
    for t in ['CASH_OUT','DEBIT','PAYMENT','TRANSFER']:
        d[f'type_{t}'] = (d['type'].str.upper() == t).astype(int)
    d['balanceDiff']  = d['oldbalanceOrg'] - d['newbalanceOrig']
    d['amount_ratio'] = d['amount'] / d['oldbalanceOrg'].replace(0, np.nan)
    d['amount_ratio'].fillna(0, inplace=True)
    for f in FEATURES:
        if f not in d.columns: d[f] = 0
    return d

df_prep = prep_batch(batch)
X = df_prep[FEATURES].values.astype(float)
X_s = scaler.transform(X)
probs = model.predict_proba(X_s)[:, 1]
preds = (probs >= meta['decision_threshold']).astype(int)

batch['fraud_probability'] = np.round(probs, 4)
batch['prediction']        = preds
batch['correct']           = (preds == batch['isFraud']).map({True:'✅',False:'❌'})

print(batch[['type','amount','isFraud','fraud_probability','prediction','correct']].to_string(index=False))
acc = (preds == batch['isFraud']).mean()
print(f'\nBatch accuracy: {acc:.0%}  ({int(acc*len(batch))}/{len(batch)})')
print('\n✅ Batch inference smoke test passed.')

---
## 6. Model Performance Summary

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

with open('outputs/metrics/model_comparison.json') as f:
    comp = json.load(f)
comp_df = pd.DataFrame(comp)

display(comp_df[['Model','Test Precision','Test Recall','Test F1','Test AUC-ROC','Test Avg Prec']])

# Quick grouped bar chart
metrics = ['Test Precision','Test Recall','Test F1','Test AUC-ROC']
x       = np.arange(len(metrics))
width   = 0.18
colors  = ['#49B6E5','#263D5B','#D97706','#DC2626']

fig, ax = plt.subplots(figsize=(10, 4))
for i, (_, row) in enumerate(comp_df.iterrows()):
    vals = [row[m] for m in metrics]
    ax.bar(x + i*width, vals, width, label=row['Model'], color=colors[i], edgecolor='white')
ax.set_xticks(x + width*1.5)
ax.set_xticklabels([m.replace('Test ','') for m in metrics])
ax.set_ylim(0, 1.1)
ax.set_title('Model Comparison — Test Metrics', fontweight='bold', color='#263D5B')
ax.legend(fontsize=8)
for sp in ax.spines.values(): sp.set_visible(False)
plt.tight_layout()
plt.savefig('outputs/deployment/model_comparison_deploy.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved → outputs/deployment/model_comparison_deploy.png')

---
## 7. Deployment File Checklist

In [ ]:
DEPLOY_FILES = [
    ('app.py',                                  'Main Streamlit dashboard'),
    ('rag_module.py',                           'RAG pipeline (existing — not modified)'),
    ('requirements.txt',                        'Deployment dependencies'),
    ('.gitignore',                              'Git exclusions (secrets, caches)'),
    ('.streamlit/config.toml',                  'Streamlit theme & server config'),
    ('.streamlit/secrets.toml',                 'API key storage (DO NOT COMMIT)'),
    ('outputs/deployment/model.pkl',            'Deployed XGBoost model'),
    ('outputs/deployment/scaler.pkl',           'Feature scaler'),
    ('outputs/deployment/model_meta.json',      'Model metadata & thresholds'),
    ('outputs/deployment/feature_names.json',   'Feature list'),
    ('outputs/models/xgboost.pkl',              'XGBoost (comparison)'),
    ('outputs/models/random_forest.pkl',        'Random Forest (comparison)'),
    ('outputs/models/neural_network.pkl',       'Neural Network (comparison)'),
    ('outputs/models/logistic_regression.pkl',  'Logistic Regression (comparison)'),
    ('outputs/metrics/model_comparison.json',   'Metrics dashboard data'),
    ('outputs/ablation/ablation_results.csv',   'Ablation study results'),
    ('chroma_db/',                              'Vector DB (SBP regulatory corpus)'),
    ('SBP_Documents/',                          'Source PDFs (optional — heavy)'),
]

print(f'{"File/Directory":<55}  {"Present":<8}  Description')
print('-'*100)
all_present = True
for path, desc in DEPLOY_FILES:
    present = os.path.exists(path)
    if not present: all_present = False
    icon = '✅' if present else '❌'
    print(f'{path:<55}  {icon:<8}  {desc}')

print()
print('🎉 All deployment files present!' if all_present
      else '⚠️  Some files missing — see ❌ rows.')

In [ ]:
print('=' * 55)
print('  TrustGuard Deployment Notebook — ALL CELLS PASSED')
print('  Ready for Streamlit Cloud deployment.')
print('=' * 55)